In [45]:
import torch
import torch.nn as nn
import torch.optim
import torchvision
import torch.nn.functional as F

In [51]:
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST('data/', train=True, download=True, transform=transform)
test_dataset = torchvision.datasets.MNIST('data/', train=False, transform=transform)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32)

In [52]:
train_loader

In [53]:
for batch_idx, (data, target) in enumerate(train_loader):
    break

In [87]:
import torch
np.random.seed(0)
torch.manual_seed(0)


model = SoftmaxRegression(4, 3)
X = np.random.randn(5, 4)
y = np.random.randint(3, size=5)
yhat = model(torch.Tensor(X))
# https://stackoverflow.com/questions/55675345/should-i-use-softmax-as-output-when-using-cross-entropy-loss-in-pytorch
criterion = torch.nn.NLLLoss() #torch.nn.CrossEntropyLoss()
loss = criterion(yhat, torch.tensor(y))
loss.backward()
yhat, model.linear.weight.grad, model.linear.weight.grad.shape

(tensor([[-2.1589, -0.4397, -1.4258],
         [-1.2080, -1.1637, -0.9445],
         [-1.7169, -0.5830, -1.3388],
         [-1.2486, -0.9753, -1.0905],
         [-0.8292, -1.6110, -1.0108]], grad_fn=<LogSoftmaxBackward>),
 tensor([[ 0.0445, -0.0864,  0.0459, -0.0805],
         [-0.2768,  0.1528, -0.1383,  0.0148],
         [ 0.2323, -0.0663,  0.0923,  0.0657]]),
 torch.Size([3, 4]))

In [88]:
import numpy as np

def softmax(x):
    z = x - np.max(x, axis=1, keepdims=True)
    e_z = np.exp(z)
    return e_z / np.sum(e_z, axis=1, keepdims=True)


w = np.array(model.linear.weight.tolist())
n = X.shape[0]
z = np.dot(X, w.T)
yhat2 = softmax(z)
yoh = np.zeros((n, 3))
for i in range(n):
    yoh[i, y[i]] = 1.
# L8.8 Softmax Regression Derivatives for Gradient Descent
# https://www.youtube.com/watch?v=aeM-fmcdkXU
# 17:50
dw = -1 * (1./n) * (np.dot(X.T, (yoh - yhat2))).T
yhat2, dw, dw.shape

(array([[0.11544643, 0.64423474, 0.24031883],
        [0.29878868, 0.31232513, 0.38888619],
        [0.17961868, 0.55822716, 0.26215416],
        [0.28689501, 0.3770632 , 0.3360418 ],
        [0.43639348, 0.19969323, 0.36391329]]),
 array([[ 0.04452056, -0.08642274,  0.04591839, -0.08049522],
        [-0.2768328 ,  0.15276102, -0.13825238,  0.01478535],
        [ 0.23231225, -0.06633828,  0.09233398,  0.06570987]]),
 (3, 4))

In [109]:
class Linear:

    def __init__(self, in_features, out_features):
        self.weight = np.zeros((out_features, in_features))
        self.cache = {}
        self.grads = {}

    def forward(self, x):
        self.cache["x"] = x
        return np.dot(x, self.weight.T)

    def backward(self, dy):
        self.grads["db"] = np.sum(dy, axis=0)
        self.grads["dw"] = np.dot(dy.T, self.cache["x"])
        dx = np.dot(dy, self.weight)
        return dx

class Softmax:

    def __init__(self):
        self.cache = {}

    def forward(self, x):
        # assumes a matrix input
        m = np.max(x, axis=-1, keepdims=True)
        e_z = np.exp(x - m)
        y = e_z / np.sum(e_z, axis=-1, keepdims=True)
        self.cache["y"] = y
        return y

    def backward(self, dy):
        dx = []
        for i in range(self.cache["y"].shape[0]):
            yi = self.cache["y"][i, :].reshape((-1, 1))
            dyidxi = np.diagflat(yi) - np.dot(yi, yi.T)
            dxi = np.dot(dy[i, :], dyidxi)
            dx.append(dxi)
        dx = np.array(dx)
        return dx


class LogSoftmax:

    def __init__(self):
        self.cache = {}

    def forward(self, x):
        # assumes a matrix input
        m = np.max(x, axis=-1, keepdims=True)
        z = x - m
        y = z - np.log(np.sum(np.exp(z), axis=-1, keepdims=True))
        self.cache["x"] = x
        return y

    def backward(self, dy):
        # I have to think more about the stability of the backward pass.
        # Right now, I just use the softmax gradient.
        softmax = Softmax()
        y = softmax.forward(self.cache["x"])
        # Multiply (1./y) because we're getting the gradient
        # of log softmax.
        dx = softmax.backward(dy * (1./y))
        return dx
    
class SoftmaxModel:

    def __init__(self):
        self.fc = Linear(784, 10)
        self.log_softmax = LogSoftmax()

    def forward(self, x):
        z = self.fc.forward(x)
        a = self.log_softmax.forward(z)
        return a

    def backward(self, dy):
        da = self.log_softmax.backward(dy)
        dz = self.fc.backward(da)
        return dz

model3 = SoftmaxModel()
model3.fc.weight = np.array(model.linear.weight.tolist())
log_yhat3 = model3.forward(X)
yhat3 = np.exp(log_yhat3)
yhat3.shape

dy = (-1./n) * yoh * (1./yhat3)
da = []
for i in range(yhat3.shape[0]):
    yi = yhat3[i, :].reshape((-1, 1))
    dyidxi = np.diagflat(yi) - np.dot(yi, yi.T)
    dxi = np.dot(dy[i, :], dyidxi)
    da.append(dxi)
da = np.array(da)

yhat3, np.dot(da.T, X)

(array([[0.11544643, 0.64423474, 0.24031883],
        [0.29878868, 0.31232513, 0.38888619],
        [0.17961868, 0.55822716, 0.26215416],
        [0.28689501, 0.3770632 , 0.3360418 ],
        [0.43639348, 0.19969323, 0.36391329]]),
 array([[ 0.04452056, -0.08642274,  0.04591839, -0.08049522],
        [-0.2768328 ,  0.15276102, -0.13825238,  0.01478535],
        [ 0.23231225, -0.06633828,  0.09233398,  0.06570987]]))